In [88]:
%pip install pandas pyspark

Note: you may need to restart the kernel to use updated packages.


In [89]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, input_file_name, regexp_extract, upper, when, abs as spark_abs, sum as spark_sum, to_date, month, year, date_format, round as spark_round
from functools import reduce

from pyspark.sql import SparkSession

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

# java 17:
# sudo apt install openjdk-17-jdk
# JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64 poetry run pyspark

spark = SparkSession.builder.appName("BankTransactions").getOrCreate()



In [90]:
# 1. Read all CSVs at once
FILE_NAME = './fidelity_credit_card.csv'
# FILE_NAME = './Discover_2026_YearToDateSummary.csv'
df = spark.read.option("header", True).csv(FILE_NAME)
# df = spark.read.option("header", True).csv()

In [91]:
# 2. Normalize column names
df = (
    df.withColumnRenamed("Description", "Name")
      .withColumnRenamed("Transaction Amt", "Amount")
      .withColumnRenamed("TransactionDate", "Date")
      .withColumnRenamed("Post Date", "Date")
)

df = df.drop("Trans. Date")

df.show()

+----------+-----------+--------------------+--------------------+-------+
|      Date|Transaction|                Name|                Memo| Amount|
+----------+-----------+--------------------+--------------------+-------+
|2025-10-06|     CREDIT|AMAZON MKTPLACE P...|74692165278103764...|   7.07|
|2025-10-06|      DEBIT|7-ELEVEN 41750   ...|24034545278001006...|  -4.50|
|2025-10-06|      DEBIT|7-ELEVEN 41750   ...|24034545278001006...|  -5.11|
|2025-10-06|      DEBIT|7-ELEVEN 41750   ...|24034545278001006...| -50.45|
|2025-10-06|      DEBIT|PY *JEREMIAHS ICE...|24445005278500848...| -13.23|
|2025-10-06|      DEBIT|SQ *BROOX COOX CA...|24692165277102857...|  -6.33|
|2025-10-06|      DEBIT|SQ *GEIER?S SAUSA...|24692165277102857...| -19.00|
|2025-10-06|      DEBIT|APPLE.COM/BILL   ...|24692165276101584...|  -1.99|
|2025-10-08|      DEBIT|SQ *MYUNG GA     ...|24692165280105778...| -30.39|
|2025-10-08|      DEBIT|H MART ORLANDO   ...|24431065280302931...|-121.98|
|2025-10-10|      DEBIT|S

In [92]:

# drop memo column
df = df.drop("Memo")
# only display money spent
if 'Transaction' in df.columns:
    df = df.filter(col("Transaction") == "DEBIT")
    


match FILE_NAME:
    case './fidelity_credit_card.csv':
        # Specific processing for Fidelity CSV
        df = df.withColumn("Date", to_date(col("Date")))
        df = df.withColumn(
            "Amount",
            spark_round(col("Amount").cast("double"), 2)
        )
        pass
    case './Discover_2026_YearToDateSummary.csv':
        # Specific processing for Discover CSV
        # 
        df = df.withColumn("Date", to_date(col("Date"), "MM/dd/yyyy"))
        df = df.withColumn(
            "Amount",
            spark_round(col("Amount").cast("double"), 2)
        ).filter(col("Amount") > 0)
        pass
    case _:
        raise FileNotFoundError("Please provide a valid CSV file.")

# cast amount to double and take absolute value
df = df.withColumn("Amount", spark_abs(col("Amount")))

# df.show()
df.toPandas()
# df.head(200)


,Date,Transaction,Name,Amount
0,2025-10-06,DEBIT,7-ELEVEN 41750 LAKELAND FL,4.50
1,2025-10-06,DEBIT,7-ELEVEN 41750 LAKELAND FL,5.11
2,2025-10-06,DEBIT,7-ELEVEN 41750 LAKELAND FL,50.45
3,2025-10-06,DEBIT,PY *JEREMIAHS ICE BRA BRADENTON FL,13.23
4,2025-10-06,DEBIT,SQ *BROOX COOX CAFE Palmetto FL,6.33
...,...,...,...,...
289,2026-04-27,DEBIT,SAMS CLUB #6189 APOPKA FL,52.80
290,2026-04-27,DEBIT,SAM'S CLUB #6189 APOPKA FL,108.98
291,2026-04-27,DEBIT,WM SUPERCENTER #4588 321-354-2096 FL,34.52
292,2026-04-27,DEBIT,PP*TUTTI FRUTTI NURSER SAINT CLOUD FL,4.00


In [93]:
# 3. Extract bank name from file path
df = df.withColumn("file_path", input_file_name())

df.select("file_path").head(1)

[Row(file_path='file:///home/leo_zhang/Documents/GitHub/Tools/python_tools/transactions_visualizer/fidelity_credit_card.csv')]

In [94]:
# add a column
df = df.withColumn(
    "Bank",
    regexp_extract(col("file_path"), r"/([^/]+)\.csv$", 1)
).drop("file_path")

df.select("Bank").head(1)

[Row(Bank='fidelity_credit_card')]

In [95]:
# 4. Define categories
categories = {
    "Income": ["PAYROLL", "DIRECT DEP", "VENMO", "REFUND", ],
    "Housing": ["RENT", "MORTGAGE", "ELECTRIC", "WATER", "GAS", ],
    "Transportation & Gas": ["SHELL", "EXXON", "UBER", "LYFT", "7-ELEVEN", "Sam's Club Gas", "SAMS FUEL", ],
    "Food & Dining": ["STARBUCKS", "DOMINOS", "DOORDASH", "TACO", "MCDONALDS", ],
    "Grocery": ["COSTCO", "KROGER", "TRADER JOE", "WALMART", "ALDI", "WM SUPERCENTER", "BRAVO", "WHOLEFDS", "Enson Market Inc", "PUBLIX", "LOTTE PLAZA", "SAMS STORE"],
    "Entertainment": ["NETFLIX", "SPOTIFY", "DISNEY", ],
    "Health & Medical": ["CVS", "WALGREENS", "KAISER", ],
    "Shopping & Personal": ["AMAZON", "BESTBUY", "MACY", ],
    "Travel": ["HOTEL", "AIRBNB", "DELTA", ],
    "Financial / Investments": ["CREDIT", "IRA", "401K", "LOAN", ]
}

# 5. Build categorization logic (Spark version of your function)
category_expr = None

for category, keywords in categories.items():
    condition = col("Name").isNull()
    for keyword in keywords:
        keyword_cond = upper(col("Name")).contains(keyword)
        condition = keyword_cond if condition is None else (condition | keyword_cond)
    
    if category_expr is None:
        category_expr = when(condition, category)
    else:
        category_expr = category_expr.when(condition, category)

# Default fallback
category_expr = category_expr.otherwise("Miscellaneous")
# category_expr


In [ ]:
df = df.withColumn("Category", category_expr)

df = df.withColumn("Month", date_format(col("Date"), "yyyy-MM"))


# 7. Show result
# df.tail(20)
df.groupBy("Category", "Month")\
.agg(
    spark_sum("Amount").alias("TotalAmount"))\
.orderBy("Month")\
.filter(col("Category") == "Grocery")\
.show()

+--------+-------+------------------+
|Category|  Month|       TotalAmount|
+--------+-------+------------------+
| Grocery|2025-10|             89.95|
| Grocery|2025-11|            256.34|
| Grocery|2025-12|411.68999999999994|
| Grocery|2026-03|            504.99|
| Grocery|2026-04|            676.77|
+--------+-------+------------------+



26/04/30 15:08:33 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 47175346 ms exceeds timeout 120000 ms
26/04/30 15:08:33 WARN SparkContext: Killing executors is not supported by current scheduler.
26/04/30 15:08:36 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint